In [4]:
# --- 1. 导入依赖 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime
import pandas as pd

# --- 2. 日志与环境设置 ---
def setup_logging(log_file='training_regression_cofrnetdl.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger(); root_logger.handlers = []; root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w'); file_handler.setFormatter(log_formatter); root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout); console_handler.setFormatter(log_formatter); root_logger.addHandler(console_handler)

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. 数据加载与处理 ---
def get_regression_data_loaders(batch_size=128, test_size=0.3):
    """准备Boston房价回归数据集"""
    logging.info("正在准备Boston房价回归数据集...")

    # 加载Boston房价数据集
    df = pd.read_csv('/kaggle/input/the-boston-houseprice-data/boston.csv')
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)

    # 特征和目标变量
    X = df.drop(['MEDV'], axis=1)
    y = df['MEDV']

    # 特征标准化
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # 目标变量标准化
    y = (y - y.mean()) / y.std()

    # 划分训练测试集
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

    # 转换为PyTorch张量
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train.values)
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test.values)

    # 创建DataLoader
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    test_dataset = torch.utils.data.TensorDataset(X_test, y_test)

    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    logging.info("Boston房价回归数据集准备完毕。")
    logging.info(f"训练样本数: {len(train_dataset)}, 测试样本数: {len(test_dataset)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")

    return trainloader, testloader, X_train.shape[1]

# --- 4. CoFrNet-DL回归模型定义 ---
class SingleLadder(nn.Module):
    """单特征梯子模块（回归版）"""
    def __init__(self, depth, epsilon=0.1):
        super(SingleLadder, self).__init__()
        self.depth = depth
        self.epsilon = epsilon
        self.weights = nn.Parameter(torch.randn(depth + 1) * 0.1)

    def safe_reciprocal(self, x):
        sign = torch.sign(x)
        abs_x = torch.abs(x)
        safe_abs = torch.clamp(abs_x, min=self.epsilon)
        return sign / safe_abs

    def forward(self, x_j):
        if self.depth == 0:
            return self.weights[0] * x_j
        result = self.weights[self.depth] * x_j
        for k in range(self.depth - 1, 0, -1):
            denominator = self.weights[k] * x_j + self.safe_reciprocal(result)
            result = denominator
        final_result = self.weights[0] * x_j + self.safe_reciprocal(result)
        return final_result

class FullLadder(nn.Module):
    """全特征梯子模块（回归版）"""
    def __init__(self, input_dim, depth, epsilon=0.1):
        super(FullLadder, self).__init__()
        self.input_dim = input_dim
        self.depth = depth
        self.epsilon = epsilon
        self.weights = nn.Parameter(torch.randn(depth + 1, input_dim) * 0.1)

    def safe_reciprocal(self, x):
        sign = torch.sign(x)
        abs_x = torch.abs(x)
        safe_abs = torch.clamp(abs_x, min=self.epsilon)
        return sign / safe_abs

    def forward(self, x):
        if self.depth == 0:
            return F.linear(x, self.weights[0])
        result = F.linear(x, self.weights[self.depth])
        for k in range(self.depth - 1, 0, -1):
            a_k = F.linear(x, self.weights[k])
            denominator = a_k + self.safe_reciprocal(result)
            result = denominator
        a_0 = F.linear(x, self.weights[0])
        final_result = a_0 + self.safe_reciprocal(result)
        return final_result

class CoFrNetDLRegression(nn.Module):
    """CoFrNet-DL回归模型"""
    def __init__(self, input_dim, output_dim=1, num_full_ladders=10, max_diag_ladder_depth=8, max_full_ladder_depth=8, epsilon=0.1):
        super(CoFrNetDLRegression, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_full_ladders = num_full_ladders
        self.max_diag_ladder_depth = max_diag_ladder_depth
        self.max_full_ladder_depth = max_full_ladder_depth

        # 对角梯子
        self.diag_ladders = nn.ModuleList()
        for i in range(input_dim):
            depth = min(max_diag_ladder_depth, max(2, 5 + (i * 2) % 6))
            self.diag_ladders.append(SingleLadder(depth, epsilon))

        # 全连接梯子
        self.full_ladders = nn.ModuleList()
        for i in range(num_full_ladders):
            depth = 2 + int((i / max(1, num_full_ladders - 1)) * (max_full_ladder_depth - 2))
            depth = min(max_full_ladder_depth, max(2, depth))
            self.full_ladders.append(FullLadder(input_dim, depth, epsilon))

        # 最终线性层
        combined_input_dim = input_dim + num_full_ladders
        self.output_layer = nn.Linear(combined_input_dim, output_dim)
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, x):
        # 对角梯子输出
        diag_outputs = []
        for j in range(self.input_dim):
            x_j = x[:, j]
            ladder_output = self.diag_ladders[j](x_j)
            diag_outputs.append(ladder_output.unsqueeze(1))
        diag_features = torch.cat(diag_outputs, dim=1)

        # 全连接梯子输出
        full_outputs = []
        for ladder in self.full_ladders:
            ladder_output = ladder(x)
            full_outputs.append(ladder_output.unsqueeze(1))
        full_features = torch.cat(full_outputs, dim=1)

        # 拼接所有特征
        combined_features = torch.cat([diag_features, full_features], dim=1)
        output = self.output_layer(combined_features)
        return output.squeeze(1)  # 输出一维

# --- 5. 训练与验证流程 ---
def train_regression_cofrnetdl(model, trainloader, testloader, hparams, log_filepath, model_save_path):
    criterion = nn.MSELoss()
    best_test_loss = float('inf')
    best_model_state = None

    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "results_per_epoch": []
    }

    logging.info("="*30)
    logging.info("  启动 CoFrNet-DL (回归版) 训练流程")
    logging.info("="*30)

    epochs = hparams['epochs']
    optimizer = optim.Adam(model.parameters(), lr=hparams['learning_rate_adam'], weight_decay=hparams['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.float().to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        epoch_avg_loss = running_loss / len(trainloader)
        scheduler.step()
        logging.info(f"Epoch {epoch+1}/{epochs} | 训练MSE损失: {epoch_avg_loss:.4f}")

        # 验证集评估
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = model(inputs)
                test_loss += criterion(outputs, targets).item()
        avg_test_loss = test_loss / len(testloader)
        log_data["results_per_epoch"].append({
            "epoch": epoch+1,
            "train_loss": epoch_avg_loss,
            "test_loss": avg_test_loss
        })
        logging.info(f"Epoch {epoch+1} | 测试MSE损失: {avg_test_loss:.4f}")

        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            best_model_state = model.state_dict()
            logging.info(f"*** 新的最佳MSE损失: {best_test_loss:.4f}，在第 {epoch+1} 轮达到。 ***")

        with open(log_filepath, 'w') as f:
            json.dump(log_data, f, indent=4)

        if best_test_loss < hparams['loss_target']:
            logging.info(f"目标损失 {hparams['loss_target']} 已达到。")
            break

    if best_model_state:
        logging.info(f"训练流程结束。保存最佳模型到 {model_save_path}")
        torch.save({
            'model_state_dict': best_model_state,
            'best_loss': best_test_loss,
            'hyperparameters': hparams
        }, model_save_path)
    else:
        logging.error("没有训练出任何有效模型，不进行保存。")

    return best_model_state

# --- 6. 主流程 ---
if __name__ == '__main__':
    # --- 文件与超参数定义 ---
    LOG_FILE = 'training_regression_cofrnetdl.log'
    JSON_LOG_FILE = 'training_regression_cofrnetdl_log.json'
    MODEL_SAVE_PATH = 'regression_cofrnetdl.pth'

    HYPERPARAMETERS = {
        "epochs": 200,
        "loss_target": 0.1,  # MSE目标值
        "num_full_ladders": 10,
        "max_diag_ladder_depth": 8,
        "max_full_ladder_depth": 8,
        "learning_rate_adam": 1e-3,
        "weight_decay": 1e-5,
    }

    setup_logging(LOG_FILE)
    logging.info(f"回归实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    trainloader, testloader, input_dim = get_regression_data_loaders()

    # 创建CoFrNet-DL回归模型
    cofrnetdl_reg = CoFrNetDLRegression(
        input_dim=input_dim,
        output_dim=1,
        num_full_ladders=HYPERPARAMETERS['num_full_ladders'],
        max_diag_ladder_depth=HYPERPARAMETERS['max_diag_ladder_depth'],
        max_full_ladder_depth=HYPERPARAMETERS['max_full_ladder_depth']
    ).to(device)

    start_time = time.time()
    best_state_dict = train_regression_cofrnetdl(
        cofrnetdl_reg, trainloader, testloader,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")

    # 加载并验证保存的最佳模型
    if best_state_dict:
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的最佳回归模型")
        logging.info("="*30)

        verification_model = CoFrNetDLRegression(
            input_dim=input_dim, output_dim=1,
            num_full_ladders=HYPERPARAMETERS['num_full_ladders'],
            max_diag_ladder_depth=HYPERPARAMETERS['max_diag_ladder_depth'],
            max_full_ladder_depth=HYPERPARAMETERS['max_full_ladder_depth']
        )
        checkpoint = torch.load(MODEL_SAVE_PATH)
        verification_model.load_state_dict(checkpoint['model_state_dict'])
        verification_model.to(device)
        verification_model.eval()

        logging.info(f"成功加载模型，保存的MSE损失: {checkpoint['best_loss']:.4f}")

        total_params = sum(p.numel() for p in verification_model.parameters() if p.requires_grad)
        logging.info(f"模型总参数数量: {total_params}")

        test_loss = 0.0
        criterion = nn.MSELoss()
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = verification_model(inputs)
                test_loss += criterion(outputs, targets).item()

        avg_test_loss = test_loss / len(testloader)
        logging.info(f"加载后的模型在测试集上的MSE损失: {avg_test_loss:.4f}")

        assert abs(avg_test_loss - checkpoint['best_loss']) < 1e-3, "验证失败！"
        logging.info("验证成功！回归模型可以被正确保存和复用。")
    else:
        logging.info("未能成功训练和保存回归模型")

2025-08-21 12:29:12,316 [INFO ]  回归实验超参数: {
    "epochs": 200,
    "loss_target": 0.1,
    "num_full_ladders": 10,
    "max_diag_ladder_depth": 8,
    "max_full_ladder_depth": 8,
    "learning_rate_adam": 0.001,
    "weight_decay": 1e-05
}
2025-08-21 12:29:12,317 [INFO ]  正在准备Boston房价回归数据集...
2025-08-21 12:29:12,332 [INFO ]  Boston房价回归数据集准备完毕。
2025-08-21 12:29:12,334 [INFO ]  训练样本数: 354, 测试样本数: 152
2025-08-21 12:29:12,334 [INFO ]  特征维度: 13, 目标维度: 1
2025-08-21 12:29:12,340 [INFO ]  ==============================
2025-08-21 12:29:12,342 [INFO ]    启动 CoFrNet-DL (回归版) 训练流程
2025-08-21 12:29:12,343 [INFO ]  ==============================
2025-08-21 12:29:12,439 [INFO ]  Epoch 1/200 | 训练MSE损失: 83.1774
2025-08-21 12:29:12,457 [INFO ]  Epoch 1 | 测试MSE损失: 66.4503
2025-08-21 12:29:12,458 [INFO ]  *** 新的最佳MSE损失: 66.4503，在第 1 轮达到。 ***
2025-08-21 12:29:12,551 [INFO ]  Epoch 2/200 | 训练MSE损失: 78.6464
2025-08-21 12:29:12,568 [INFO ]  Epoch 2 | 测试MSE损失: 60.9880
2025-08-21 12:29:12,570 [INFO ]  *** 新的最佳

AssertionError: 验证失败！